In [2]:
import sys
sys.path.append("..")

from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch
import bitsandbytes as bnb
from src.utils import compute_metrics, set_global_seed
from src.data import load_and_preprocess_dataset
import wandb
import os
os.environ["WANDB_DIR"] = "/wandb"

In [3]:
set_global_seed(42)
model_name = "distilbert-base-uncased"

In [4]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    llm_int8_skip_modules=["classifier", "pre_classifier", "embeddings", "LayerNorm"]  
)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    device_map="auto",
    quantization_config=quantization_config,
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
model = prepare_model_for_kbit_training(model)

In [6]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_lin", "k_lin", "v_lin", "out_lin"],  
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS")

In [7]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
dataset = load_and_preprocess_dataset(tokenizer)

In [8]:
model = get_peft_model(model, lora_config)

In [10]:
training_args = TrainingArguments(
    output_dir="../results/PEFT_QLoRA",
    per_device_train_batch_size=16,  
    per_device_eval_batch_size=32,   
    num_train_epochs=5,              
    learning_rate=5e-4,              
    weight_decay=0.01,               
    warmup_steps=500,                
    lr_scheduler_type="cosine",      
    eval_strategy="epoch",     
    save_strategy="epoch",           
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    fp16=True,                       
    logging_dir="./logs",
    logging_steps=20,
    report_to="wandb",
    run_name="PEFT_QLoRA",
)

In [11]:
wandb.init(
    project="Fine-tuning-and-Optimizing-DistilBERT-for-Sentiment-Classification",   
    name="PEFT_QLoRA",                    
    config=training_args.to_dict()           
)

wandb: Currently logged in as: shivamsinghml (shivamsingh-ml) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [12]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    compute_metrics=compute_metrics,
)

trainer.train()

No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
c:\Users\shiva\OneDrive\Desktop\Projects\Fine-tuning-and-Optimizing-DistilBERT-for-Sentiment-Classification\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py:838: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.273900,0.285655,0.887615
2,0.198300,0.340238,0.892202
3,0.190800,0.312806,0.902523
4,0.100500,0.365700,0.901376
5,0.120200,0.428984,0.899083


c:\Users\shiva\OneDrive\Desktop\Projects\Fine-tuning-and-Optimizing-DistilBERT-for-Sentiment-Classification\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py:838: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
c:\Users\shiva\OneDrive\Desktop\Projects\Fine-tuning-and-Optimizing-DistilBERT-for-Sentiment-Classification\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py:838: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can 

TrainOutput(global_step=21050, training_loss=0.1708977332449299, metrics={'train_runtime': 2365.1152, 'train_samples_per_second': 142.38, 'train_steps_per_second': 8.9, 'total_flos': 1.145761067269632e+16, 'train_loss': 0.1708977332449299, 'epoch': 5.0})

In [14]:
trainer.evaluate()

{'eval_loss': 0.31280580163002014,
 'eval_accuracy': 0.9025229357798165,
 'eval_runtime': 1.2525,
 'eval_samples_per_second': 696.23,
 'eval_steps_per_second': 22.356,
 'epoch': 5.0}